In [ ]:
import cv2import numpy as npimport shutilimport timefrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutor, as_completedfrom tqdm import tqdmDRIVE_OUT   = Path("/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors/training_final")LOCAL_FINAL = Path("/content/training_final")WORKERS     = 64

In [ ]:
import cv2import numpy as npimport timefrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutor, as_completedfrom tqdm import tqdmLOCAL_FINAL = Path("/content/training_final")NPY_DIR     = LOCAL_FINAL / "npy_cache"NPY_DIR.mkdir(exist_ok=True)CLASSES  = ["normal","trespassing","loitering","object_abandonment"]IMG_SIZE = 112CLIP_LEN = 16WORKERS  = 32MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)def preprocess_clip(clip_dir):    frames_paths = sorted(clip_dir.glob("*.jpg"))    if len(frames_paths) != CLIP_LEN: return None    frames = []    for fp in frames_paths:        img = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)        if img is None: return None        img   = cv2.resize(img, (IMG_SIZE,IMG_SIZE),                           interpolation=cv2.INTER_LINEAR)        img_f = img.astype(np.float32) / 255.0        img_3 = np.stack([img_f]*3, axis=0)        for c in range(3):            img_3[c] = (img_3[c] - MEAN[c]) / STD[c]        frames.append(img_3)    return np.stack(frames, axis=0).astype(np.float32)print("Klip listesi hazırlanıyor...")all_jobs = []for split in ["train","test"]:    for cls in CLASSES:        cls_dir = LOCAL_FINAL / split / cls        if not cls_dir.exists(): continue        for day_dir in sorted(cls_dir.iterdir()):            if not day_dir.is_dir(): continue            for clip_dir in sorted(day_dir.iterdir()):                if not clip_dir.is_dir(): continue                if len(list(clip_dir.glob("*.jpg"))) == CLIP_LEN:                    all_jobs.append((split, cls,                                     day_dir.name,                                     clip_dir.name,                                     clip_dir))print(f"Toplam klip: {len(all_jobs)}")def process_and_save(job):    split, cls, day, clip_name, clip_dir = job    out_dir  = NPY_DIR / split / cls / day    out_dir.mkdir(parents=True, exist_ok=True)    out_path = out_dir / f"{clip_name}.npy"    if out_path.exists(): return "skip"    arr = preprocess_clip(clip_dir)    if arr is None: return "error"    np.save(str(out_path), arr)    return "ok"print(f"Preprocess ({WORKERS} worker)...")t0  = time.time()res = {"ok":0,"skip":0,"error":0}with ThreadPoolExecutor(max_workers=WORKERS) as ex:    futures = {ex.submit(process_and_save,j):j for j in all_jobs}    for f in tqdm(as_completed(futures), total=len(all_jobs)):        res[f.result()] += 1print(f"✓ {time.time()-t0:.0f}sn  {res}")print("\n── NPY Cache Özet ──────────────────────────────────────")total_size = 0for split in ["train","test"]:    for cls in CLASSES:        cls_dir = NPY_DIR/split/cls        if not cls_dir.exists(): continue        npys = list(cls_dir.rglob("*.npy"))        size = sum(f.stat().st_size for f in npys)        total_size += size        print(f"  {split}/{cls:<22} {len(npys):>5} klip  {size/1e6:>8.1f}MB")print(f"\n  Toplam: {total_size/1e9:.2f} GB")sample = list((NPY_DIR/"train"/"normal").rglob("*.npy"))[0]arr    = np.load(str(sample))print(f"  Shape: {arr.shape}  dtype:{arr.dtype}")print(f"  Min:{arr.min():.3f}  Max:{arr.max():.3f}")

In [ ]:
from pathlib import PathDRIVE_OUT   = Path("/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors/training_final")LOCAL_FINAL = Path("/content/training_final")print("── Drive vs Locale Karşılaştırma ───────────────────────")for split in ["train","test"]:    for cls in ["normal","trespassing","loitering","object_abandonment"]:        drive_clips = len(list((DRIVE_OUT/split/cls).rglob("frame_001.jpg"))) \                      if (DRIVE_OUT/split/cls).exists() else 0        local_clips = len(list((LOCAL_FINAL/split/cls).rglob("frame_001.jpg"))) \                      if (LOCAL_FINAL/split/cls).exists() else 0        flag = "✓" if drive_clips==local_clips else "✗ FARK!"        print(f"  {split}/{cls:<22} Drive:{drive_clips:>4}  "              f"Local:{local_clips:>4}  {flag}")

In [ ]:
import torchimport torch.nn as nnimport torch.optim as optimfrom torchvision.models.video import r3d_18, R3D_18_Weightsfrom torch.amp import GradScaler, autocastfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subsetfrom collections import defaultdictfrom sklearn.metrics import classification_report, confusion_matrix, f1_scoreimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport time, shutil, randomfrom pathlib import PathCLASSES   = ["normal","trespassing","loitering","object_abandonment"]N_CLASSES = 4DEVICE    = torch.device("cuda")LOCAL_FINAL = Path("/content/training_final")NPY_DIR     = LOCAL_FINAL / "npy_cache"SAVE_LOCAL = Path("/content/cnn8_checkpoints")SAVE_LOCAL.mkdir(exist_ok=True)SAVE_DRIVE = Path("/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors/training_final/cnn8_model")SAVE_DRIVE.mkdir(parents=True, exist_ok=True)torch.backends.cudnn.benchmark = Truerandom.seed(42)np.random.seed(42)EPOCHS_FROZEN   = 5EPOCHS_FINETUNE = 45EPOCHS_TOTAL    = EPOCHS_FROZEN + EPOCHS_FINETUNEPATIENCE        = 10BATCH_SIZE      = 32LR_FC           = 1e-3LR_FT           = 1e-4WEIGHT_DECAY    = 2e-4DROPOUT         = 0.4LABEL_SMOOTH    = 0.1VAL_RATIO       = 0.20def is_winter(day): return 202101 <= int(day[:6]) <= 202103print("── CNN8 Hiperparametreler ───────────────────────────────")print(f"  Maskeleme    : YOK")print(f"  Batch        : {BATCH_SIZE}")print(f"  LR_FC/FT     : {LR_FC}/{LR_FT}")print(f"  Dropout      : {DROPOUT}")print(f"  LabelSmooth  : {LABEL_SMOOTH}")print(f"  WeightDecay  : {WEIGHT_DECAY}")print(f"  Val ratio    : {VAL_RATIO}")print(f"  Epochs       : {EPOCHS_TOTAL}")print(f"  Patience     : {PATIENCE}")

In [ ]:
import torchimport torch.nn as nnfrom torchvision.models.video import r3d_18, R3D_18_Weightsfrom torch.amp import autocastfrom torch.utils.data import Dataset, DataLoaderfrom sklearn.metrics import (classification_report, confusion_matrix,                              f1_score, accuracy_score)import numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathimport randomCLASSES   = ["normal", "trespassing", "loitering", "object_abandonment"]N_CLASSES = 4DEVICE    = torch.device("cuda")NPY_DIR    = Path("/content/training_final/npy_cache")MODEL_PATH = Path("/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors/training_final/cnn9_model/best_model.pth")SAVE_DIR   = Path("/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors/training_final/cnn9_model")random.seed(42)np.random.seed(42)

In [ ]:
import torchimport torch.nn as nnfrom torchvision.models.video import r3d_18, R3D_18_Weightsfrom torch.amp import autocastimport numpy as npimport matplotlib.pyplot as pltimport cv2, random, timefrom pathlib import PathCLASSES    = ["normal", "trespassing", "loitering", "object_abandonment"]N_CLASSES  = 4DEVICE     = torch.device("cuda")LTD_DIR    = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")MODEL_PATH = Path("/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors/training_final/cnn9_model/best_model.pth")MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1,1)STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1,1)CLIP_FRAMES = 16random.seed(int(time.time()))COLORS = {    "normal":             "

 BURAYA KADAR SİNGLE-LABEL EĞİTİM YAPILDI ASAĞIDA DA MULTİLABEL EĞİTİM YAPILACAK 4 EK SINIF İLE TEMPORAL COMPOSİTE KULLANIP SOFTMAX  YERİNE SİGMOİD KULLANCAĞIZ


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, randomfrom pathlib import PathLOCAL_BASE = Path("/content/training_final")ALL_CLASSES = [    "normal","trespassing","loitering","object_abandonment",    "loitering+trespassing","loitering+object_abandonment",    "trespassing+object_abandonment",    "loitering+trespassing+object_abandonment",]random.seed(42)for split in ["train","test"]:    print(f"\n{'='*50}  {split.upper()}")    for cls in ALL_CLASSES:        cls_dir = LOCAL_BASE/split/cls        if not cls_dir.exists():            print(f"  [YOK] {split}/{cls}"); continue        clips = [f.parent for f in cls_dir.rglob("frame_001.jpg")]        if not clips:            print(f"  [BOŞ] {split}/{cls}"); continue        samples = random.sample(clips, min(9, len(clips)))        n = len(clips)        fig, axes = plt.subplots(3, 3, figsize=(15, 12))        axes = axes.flatten()        for i in range(9):            ax = axes[i]            if i < len(samples):                jpgs = sorted(samples[i].glob("*.jpg"))                img  = cv2.imread(str(jpgs[min(7, len(jpgs)-1)]))                rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)                ax.imshow(rgb); ax.axis('off')                ax.set_title(samples[i].name[:20], fontsize=7, pad=2)            else:                ax.axis('off')        fig.suptitle(            f"{split.upper()} / {cls}  ({n} klip)",            fontsize=12, fontweight='bold', y=1.01        )        plt.tight_layout()        plt.show()

In [ ]:
import cv2, numpy as np, shutil, random, timefrom pathlib import Pathfrom tqdm import tqdmLOCAL_BASE = Path("/content/training_final")DRIVE_BASE = Path("/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors/training_final")random.seed(42)cls_local = LOCAL_BASE/"train"/"loitering+trespassing+object_abandonment"cls_drive = DRIVE_BASE/"train"/"loitering+trespassing+object_abandonment"def aug_gaussian(frames):    std=random.uniform(0.01,0.03)    return [(np.clip(f.astype(np.float32)/255+np.random.normal(0,std,f.shape).astype(np.float32),0,1)*255).astype(np.uint8) for f in frames]def aug_bright(frames):    return [np.clip(f.astype(np.float32)*random.uniform(0.8,1.2),0,255).astype(np.uint8) for f in frames]def aug_contrast(frames):    a=random.uniform(0.85,1.15)    return [np.clip((f.astype(np.float32)-f.mean())*a+f.mean(),0,255).astype(np.uint8) for f in frames]def aug_gamma(frames):    t=np.array([(i/255)**random.uniform(0.8,1.3)*255 for i in range(256)]).astype(np.uint8)    return [cv2.LUT(f,t) for f in frames]def aug_blur(frames): return [cv2.GaussianBlur(f,(3,3),0) for f in frames]def aug_rev(frames): return list(reversed(frames))def aug_sp(frames):    out=[]    for f in frames:        o=f.copy(); n=int(0.005*f.size)        r,c=np.random.randint(0,f.shape[0],n),np.random.randint(0,f.shape[1],n); o[r,c]=255        r,c=np.random.randint(0,f.shape[0],n),np.random.randint(0,f.shape[1],n); o[r,c]=0        out.append(o)    return outAUGS=[aug_gaussian,aug_bright,aug_contrast,aug_gamma,aug_blur,aug_rev,aug_sp]def apply_aug(frames):    for fn in random.sample(AUGS,random.randint(1,2)): frames=fn(frames)    return framesdef read_frames(d):    out=[]    for jp in sorted(d.glob("*.jpg"))[:16]:        img=cv2.imread(str(jp))        if img is not None: out.append(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))    return outdef write_clip(frames,dst):    dst.mkdir(parents=True,exist_ok=True)    for i,f in enumerate(frames):        cv2.imwrite(str(dst/f"frame_{i+1:03d}.jpg"),cv2.cvtColor(f,cv2.COLOR_RGB2BGR))def copy_to_drive(l,d):    d.mkdir(parents=True,exist_ok=True)    for jp in l.glob("*.jpg"):        dp=d/jp.name        if not dp.exists(): shutil.copy2(str(jp),str(dp))

In [ ]:
class OrigOnlyDS(Dataset):    def __init__(self):        self.samples = []        for ci, cls in enumerate(ORIG_CLASSES):            label = [0.0]*N_ORIG; label[ci] = 1.0            d = NPY_BASE/"test"/cls            if not d.exists(): continue            for npy in sorted(d.glob("*.npy")):                self.samples.append((npy, label[:]))        print(f"[test_orig] {len(self.samples)} klip")    def __len__(self): return len(self.samples)    def __getitem__(self,idx):        p,l = self.samples[idx]        return torch.from_numpy(np.load(str(p))), torch.tensor(l,dtype=torch.float32)test_orig_ds     = OrigOnlyDS()test_orig_loader = DataLoader(test_orig_ds, 32, shuffle=False,                               num_workers=2, pin_memory=True)ckpt = torch.load("/content/cnn_multilabel_final/best_model.pth")model.load_state_dict(ckpt["model_state"])_,_,all_probs,all_lbl = eval_epoch(test_orig_loader)pred_idx = all_probs.argmax(axis=1)true_idx = all_lbl.argmax(axis=1)f1_final = f1_score(true_idx, pred_idx, average='macro')print(f"Test F1 (sadece 4 orijinal sınıf): {f1_final:.4f}")print(f"\nKarşılaştırma:")print(f"  CNN9+threshold=0.72  : 0.9721")print(f"  Multi-label final    : {f1_final:.4f}")print(f"\n{classification_report(true_idx,pred_idx,target_names=ORIG_CLASSES,digits=3)}")

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics import confusion_matrix, f1_scorefrom torch.utils.data import Dataset, DataLoaderimport torchclass FullTestDS(Dataset):    def __init__(self):        self.samples = []        for ci, cls in enumerate(ALL_CLASSES):            if cls in ORIG_CLASSES:                label = [0.0]*N_ORIG; label[ORIG_CLASSES.index(cls)] = 1.0            else:                label = [float(x) for x in COMP_LABELS[cls]]            d = NPY_BASE/"test"/cls            if not d.exists(): continue            for npy in sorted(d.glob("*.npy")):                self.samples.append((npy, label[:], cls))        print(f"[full_test] {len(self.samples)} klip")    def __len__(self): return len(self.samples)    def __getitem__(self,idx):        p,l,cls = self.samples[idx]        return torch.from_numpy(np.load(str(p))), torch.tensor(l,dtype=torch.float32), clsfull_ds     = FullTestDS()full_loader = DataLoader(full_ds, 32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, cv2, torch, randomfrom torch.utils.data import Dataset, DataLoaderfrom pathlib import PathNPY_BASE   = Path("/content/npy_final")LOCAL_BASE = Path("/content/training_final")DEVICE     = torch.device("cuda")ALL_CLASSES = [    "normal","trespassing","loitering","object_abandonment",    "loitering+trespassing","loitering+object_abandonment",    "trespassing+object_abandonment",    "loitering+trespassing+object_abandonment",]ORIG_CLASSES = ["normal","trespassing","loitering","object_abandonment"]N_ORIG = 4COMP_LABELS = {    "loitering+trespassing":                    [0,1,1,0],    "loitering+object_abandonment":             [0,0,1,1],    "trespassing+object_abandonment":           [0,1,0,1],    "loitering+trespassing+object_abandonment": [0,1,1,1],}CLASS_VECTORS = {}for cls in ORIG_CLASSES:    v = [0.0]*N_ORIG; v[ORIG_CLASSES.index(cls)]=1.0    CLASS_VECTORS[cls] = vfor cls, v in COMP_LABELS.items():    CLASS_VECTORS[cls] = [float(x) for x in v]SHORT = {    "normal":"normal","trespassing":"tresp","loitering":"loiter",    "object_abandonment":"obj","loitering+trespassing":"loi+tre",    "loitering+object_abandonment":"loi+obj",    "trespassing+object_abandonment":"tre+obj",    "loitering+trespassing+object_abandonment":"loi+tre+obj",}

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as pltfrom pathlib import PathLOCAL_BASE = Path("/content/training_final")NPY_BASE   = Path("/content/npy_final")ORIG_CLASSES = ["normal","trespassing","loitering","object_abandonment"]COLORS = {"normal":"